In [1]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import itertools
from pathlib import Path

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

from datasets import load_dataset

torch.manual_seed(0)
np.random.seed(0)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("device:", device)

device: cpu


In [2]:
MODEL_ID = "openai-community/gpt2"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID
).to(device)

model.eval()

N_LAYERS = model.config.n_layer

print(
    "layers:",
    N_LAYERS
)

Loading weights:   0%|          | 0/148 [00:03<?, ?it/s]

layers: 12


In [3]:
counterfact = load_dataset(
    "azhx/counterfact",
    split="test"
)

print(counterfact)
print(counterfact[0]["requested_rewrite"])

Dataset({
    features: ['case_id', 'pararel_idx', 'requested_rewrite', 'paraphrase_prompts', 'neighborhood_prompts', 'attribute_prompts', 'generation_prompts'],
    num_rows: 2191
})
{'prompt': '{} is located in', 'relation_id': 'P30', 'subject': 'Angola', 'target_new': {'id': 'Q51', 'str': 'Antarctica'}, 'target_true': {'id': 'Q15', 'str': 'Africa'}}


In [4]:
def one_token_id(text):

    text = " " + text.strip()

    ids = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    if len(ids) != 1:
        return None

    return ids[0]

In [5]:
def one_token_id(text):

    text = " " + text.strip()

    ids = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    if len(ids) != 1:
        return None

    return ids[0]

In [6]:
def logit_margin(
    logits,
    true_id,
    alt_id
):

    return (
        logits[0, -1, true_id]
        -
        logits[0, -1, alt_id]
    ).item()

In [7]:
def subject_token_positions(
    prompt,
    subject
):

    start = prompt.find(subject)

    if start < 0:
        return None

    end = start + len(subject)

    enc = tokenizer(
        prompt,
        return_offsets_mapping=True,
        add_special_tokens=False
    )

    positions = []

    for i, (a, b) in enumerate(
        enc["offset_mapping"]
    ):

        if (
            b > start
            and
            a < end
        ):
            positions.append(i)

    return positions

In [8]:
EMBED_STD = (
    model.transformer.wte.weight
    .detach()
    .std()
    .item()
)

NOISE_SCALE = 3.0 * EMBED_STD

print(
    "embedding std:",
    EMBED_STD
)

print(
    "noise scale:",
    NOISE_SCALE
)

embedding std: 0.14369554817676544
noise scale: 0.4310866445302963


In [9]:
def corrupted_embeddings(
    input_ids,
    subj_positions,
    seed
):

    with torch.no_grad():

        emb = (
            model.transformer.wte(
                input_ids
            )
        )

        corrupted = emb.clone()

        gen = torch.Generator(
            device=device
        )

        gen.manual_seed(
            int(seed)
        )

        shape = corrupted[
            :,
            subj_positions,
            :
        ].shape

        noise = torch.randn(
            shape,
            generator=gen,
            device=device,
            dtype=corrupted.dtype
        )

        corrupted[
            :,
            subj_positions,
            :
        ] += (
            NOISE_SCALE * noise
        )

    return corrupted

In [10]:
def prepare_case(
    record,
    case_number
):

    rw = record[
        "requested_rewrite"
    ]

    subject = rw["subject"]

    prompt = rw["prompt"].format(
        subject
    )

    true_text = rw[
        "target_true"
    ]["str"]

    new_text = rw[
        "target_new"
    ]["str"]

    true_id = one_token_id(
        true_text
    )

    new_id = one_token_id(
        new_text
    )

    if (
        true_id is None
        or
        new_id is None
        or
        true_id == new_id
    ):
        return None

    positions = (
        subject_token_positions(
            prompt,
            subject
        )
    )

    if not positions:
        return None

    enc = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    )

    input_ids = (
        enc["input_ids"]
        .to(device)
    )

    attention_mask = (
        enc["attention_mask"]
        .to(device)
    )

    with torch.inference_mode():

        clean_out = model(
            input_ids=input_ids,
            attention_mask=
                attention_mask,
            use_cache=False
        )

    D_clean = logit_margin(
        clean_out.logits,
        true_id,
        new_id
    )

    # il modello deve preferire
    # chiaramente il fatto vero
    if D_clean < 1.0:
        return None

    corrupt_emb = (
        corrupted_embeddings(
            input_ids,
            positions,
            seed=100000 + case_number
        )
    )

    with torch.inference_mode():

        corrupt_out = model(
            inputs_embeds=
                corrupt_emb,
            attention_mask=
                attention_mask,
            use_cache=False
        )

    D_corr = logit_margin(
        corrupt_out.logits,
        true_id,
        new_id
    )

    TE = D_clean - D_corr

    # corruption deve aver
    # prodotto un effetto reale
    if TE < 1.0:
        return None

    return {
        "case_id": case_number,

        "prompt": prompt,
        "subject": subject,

        "true_text": true_text,
        "new_text": new_text,

        "true_id": true_id,
        "new_id": new_id,

        "input_ids":
            input_ids.detach(),

        "attention_mask":
            attention_mask.detach(),

        "subject_positions":
            positions,

        "subject_last":
            positions[-1],

        "query_pos":
            input_ids.shape[1] - 1,

        "seed":
            100000 + case_number,

        "D_clean":
            D_clean,

        "D_corr":
            D_corr,

        "TE":
            TE
    }

In [11]:
rng = np.random.default_rng(
    123
)

order = rng.permutation(
    len(counterfact)
)

cases = []

N_TOTAL = 64

for idx in order:

    case = prepare_case(
        counterfact[int(idx)],
        int(idx)
    )

    if case is not None:

        cases.append(case)

        print(
            len(cases),
            case["subject"],
            "| TE =",
            round(case["TE"], 3)
        )

    if len(cases) >= N_TOTAL:
        break

print(
    "usable facts:",
    len(cases)
)

1 PAX | TE = 3.5
2 Rudrur | TE = 2.047
3 Michel-Jean Sedaine | TE = 3.425
4 Carla Accardi | TE = 1.435
5 Slovenia | TE = 1.6
6 Pablo Picasso | TE = 1.361
7 Abu Ghosh | TE = 1.321
8 Flanders | TE = 6.334
9 BBC Radio 2 | TE = 6.306
10 Louis McLane | TE = 1.234
11 Johan Gastien | TE = 1.956
12 Otto Jelinek | TE = 1.895
13 Toyota Sprinter | TE = 1.619
14 Isabelle Caro | TE = 3.467
15 Phoebe Atwood Taylor | TE = 1.755
16 Mount St Gwinear | TE = 1.284
17 Elk State Park | TE = 2.06
18 Nasiruddin Chiragh Dehlavi | TE = 2.091
19 Michel Bouquet | TE = 3.407
20 ArcelorMittal Poland | TE = 3.546
21 IBM Network Control Program | TE = 1.86
22 Tim Mitchell | TE = 2.566
23 Bernard Giraudeau | TE = 3.569
24 Lancashire wrestling | TE = 1.036
25 Sofia | TE = 1.423
26 Salman the Persian | TE = 2.272
27 Georges Duhamel | TE = 3.088
28 Nissan GT-R LM Nismo | TE = 2.34
29 Pontecorvo | TE = 2.912
30 Gerard Mortier | TE = 4.771
31 Second Italo-Ethiopian War | TE = 9.214
32 Frank Lampard | TE = 4.512
33 Tirana 

In [12]:
train_cases = cases[:32]
test_cases = cases[32:64]

print(
    len(train_cases),
    len(test_cases)
)

32 32


In [13]:
phase5_cases = train_cases + test_cases

OLD_SUBJECTS = {
    c["subject"].strip().lower()
    for c in phase5_cases
}

OLD_FACT_KEYS = {
    (
        c["subject"].strip().lower(),
        c["prompt"].strip().lower(),
        c["true_text"].strip().lower(),
        c["new_text"].strip().lower(),
    )
    for c in phase5_cases
}

print("old subjects:", len(OLD_SUBJECTS))
print("old exact facts:", len(OLD_FACT_KEYS))

old subjects: 64
old exact facts: 64


In [14]:
fresh_manifest["subject_key"] = (
    fresh_manifest["subject"]
    .str.strip()
    .str.lower()
)

overlap_subject = fresh_manifest[
    fresh_manifest["subject_key"].isin(
        OLD_SUBJECTS
    )
]

print(
    "subject overlaps:",
    len(overlap_subject)
)

display(
    overlap_subject[
        [
            "case_id",
            "subject",
            "prompt",
            "true_text",
            "new_text",
        ]
    ]
)

NameError: name 'fresh_manifest' is not defined

In [15]:
import pandas as pd

phase5_cases = train_cases + test_cases

OLD_SUBJECTS = {
    c["subject"].strip().lower()
    for c in phase5_cases
}

phase5_seen = pd.DataFrame([
    {
        "case_id": c["case_id"],
        "subject": c["subject"],
        "subject_key": c["subject"].strip().lower(),
        "prompt": c["prompt"],
        "true_text": c["true_text"],
        "new_text": c["new_text"],
    }
    for c in phase5_cases
])

phase5_seen.to_csv(
    "phase5_seen_facts.csv",
    index=False
)

print("Phase-5 cases:", len(phase5_seen))
print("Unique subjects:", len(OLD_SUBJECTS))

phase5_seen.head()

Phase-5 cases: 64
Unique subjects: 64


,case_id,subject,subject_key,prompt,true_text,new_text
0,1508,PAX,pax,Where is PAX? It is located in,Seattle,Leeds
1,448,Rudrur,rudrur,"Rudrur, located in",India,Philippines
2,401,Michel-Jean Sedaine,michel-jean sedaine,The mother tongue of Michel-Jean Sedaine is,French,Russian
3,1985,Carla Accardi,carla accardi,Carla Accardi died at,Rome,Oslo
4,1531,Slovenia,slovenia,Slovenia belongs to the continent of,Europe,Antarctica


In [16]:
fresh_manifest = pd.read_csv(
    "phase6_fresh_manifest.csv"
)

fresh_manifest["subject_key"] = (
    fresh_manifest["subject"]
    .str.strip()
    .str.lower()
)

overlap_subject = fresh_manifest[
    fresh_manifest["subject_key"].isin(
        OLD_SUBJECTS
    )
]

print(
    "subject overlaps:",
    len(overlap_subject)
)

display(
    overlap_subject[
        [
            "case_id",
            "subject",
            "prompt",
            "true_text",
            "new_text",
        ]
    ]
)

subject overlaps: 6


,case_id,subject,prompt,true_text,new_text
7,4263,Victoria Derbyshire,"Victoria Derbyshire, of",BBC,IBM
11,11078,Everberg,"Everberg, which is located in",Belgium,Australia
20,14700,Chicago Rockets,Where is Chicago Rockets? It is located in,Chicago,Singapore
29,19180,Ely Guerra,"Ely Guerra, who is a citizen of",Mexico,Norway
51,17081,Frank Lampard,"Frank Lampard, the",midfielder,quarterback
112,17485,Louis-Paul Aujoulat,Louis-Paul Aujoulat is a native speaker of,French,Dutch


In [ ]:
subject_key = (
    record["requested_rewrite"]["subject"]
    .strip()
    .lower()
)

if subject_key in OLD_SUBJECTS:
    continue

In [ ]:
SITES = [
    (layer, role)

    for layer in range(N_LAYERS)

    for role in ["S", "Q"]
]

print(
    "sites:",
    len(SITES)
)

In [ ]:
def runtime_case(case):

    input_ids = case[
        "input_ids"
    ]

    attention_mask = case[
        "attention_mask"
    ]

    with torch.inference_mode():

        clean_out = model(
            input_ids=input_ids,
            attention_mask=
                attention_mask,
            output_hidden_states=True,
            use_cache=False
        )

    clean_states = (
        clean_out.hidden_states
    )

    corrupt_emb = (
        corrupted_embeddings(
            input_ids,
            case[
                "subject_positions"
            ],
            case["seed"]
        )
    )

    with torch.inference_mode():

        corr_out = model(
            inputs_embeds=
                corrupt_emb,
            attention_mask=
                attention_mask,
            use_cache=False
        )

    D0 = logit_margin(
        corr_out.logits,
        case["true_id"],
        case["new_id"]
    )

    return (
        clean_states,
        corrupt_emb,
        D0
    )

In [ ]:
def patched_margin(
    case,
    clean_states,
    corrupt_emb,
    sites
):

    by_layer = {}

    for layer, role in sites:

        if role == "S":
            pos = case[
                "subject_last"
            ]

        else:
            pos = case[
                "query_pos"
            ]

        clean_value = (
            clean_states[
                layer + 1
            ][
                :, pos, :
            ]
        )

        by_layer.setdefault(
            layer,
            []
        ).append(
            (
                pos,
                clean_value
            )
        )

    handles = []

    def make_hook(patches):

        def hook(
            module,
            inputs,
            output
        ):

            if isinstance(
                output,
                tuple
            ):

                h = output[0].clone()

                for pos, value in patches:

                    h[:, pos, :] = (
                        value
                    )

                return (
                    h,
                    *output[1:]
                )

            h = output.clone()

            for pos, value in patches:

                h[:, pos, :] = value

            return h

        return hook

    try:

        for layer, patches in (
            by_layer.items()
        ):

            handle = (
                model.transformer
                .h[layer]
                .register_forward_hook(
                    make_hook(
                        patches
                    )
                )
            )

            handles.append(
                handle
            )

        with torch.inference_mode():

            out = model(
                inputs_embeds=
                    corrupt_emb,
                attention_mask=
                    case[
                        "attention_mask"
                    ],
                use_cache=False
            )

        return logit_margin(
            out.logits,
            case["true_id"],
            case["new_id"]
        )

    finally:

        for h in handles:
            h.remove()

In [ ]:
single_rows = []
pair_rows = []

for c_idx, case in enumerate(
    train_cases
):

    (
        clean_states,
        corrupt_emb,
        D0
    ) = runtime_case(case)

    TE = case["TE"]

    single_D = {}

    for site in SITES:

        Di = patched_margin(
            case,
            clean_states,
            corrupt_emb,
            [site]
        )

        single_D[site] = Di

        single_rows.append({
            "case_id":
                case["case_id"],

            "layer":
                site[0],

            "role":
                site[1],

            "IE_norm":
                (Di - D0) / TE
        })

    for si, sj in (
        itertools.combinations(
            SITES,
            2
        )
    ):

        Dij = patched_margin(
            case,
            clean_states,
            corrupt_emb,
            [si, sj]
        )

        Di = single_D[si]
        Dj = single_D[sj]

        J = (
            Dij
            - Di
            - Dj
            + D0
        ) / TE

        pair_recovery = (
            Dij - D0
        ) / TE

        pair_rows.append({
            "case_id":
                case["case_id"],

            "li": si[0],
            "ri": si[1],

            "lj": sj[0],
            "rj": sj[1],

            "J_norm": J,

            "pair_recovery":
                pair_recovery
        })

    print(
        f"{c_idx+1}/"
        f"{len(train_cases)}"
        " complete"
    )

    # salvataggio progressivo
    pd.DataFrame(
        single_rows
    ).to_csv(
        "phase5_single_train.csv",
        index=False
    )

    pd.DataFrame(
        pair_rows
    ).to_csv(
        "phase5_pair_train.csv",
        index=False
    )

In [ ]:
single_df = pd.DataFrame(
    single_rows
)

pair_df = pd.DataFrame(
    pair_rows
)

site_stats = (
    single_df
    .groupby(
        ["layer", "role"]
    )
    ["IE_norm"]
    .agg(
        ["mean", "std", "count"]
    )
    .reset_index()
)

pair_stats = (
    pair_df
    .groupby(
        [
            "li", "ri",
            "lj", "rj"
        ]
    )
    ["J_norm"]
    .agg(
        ["mean", "std", "count"]
    )
    .reset_index()
)

print(
    site_stats.sort_values(
        "mean",
        ascending=False
    ).head(15)
)

print(
    pair_stats.sort_values(
        "mean",
        ascending=False
    ).head(15)
)

In [ ]:
mean_IE = {}

for _, row in site_stats.iterrows():

    site = (
        int(row["layer"]),
        row["role"]
    )

    mean_IE[site] = (
        row["mean"]
    )

In [ ]:
mean_J = {}

for _, row in pair_stats.iterrows():

    a = (
        int(row["li"]),
        row["ri"]
    )

    b = (
        int(row["lj"]),
        row["rj"]
    )

    key = tuple(
        sorted(
            [a, b]
        )
    )

    mean_J[key] = (
        row["mean"]
    )

In [ ]:
def get_J(a, b):

    key = tuple(
        sorted(
            [a, b]
        )
    )

    return mean_J.get(
        key,
        0.0
    )

In [ ]:
def select_AIE(k):

    return sorted(
        SITES,
        key=lambda s:
            mean_IE[s],
        reverse=True
    )[:k]

In [ ]:
def select_interaction(
    k
):

    selected = []

    remaining = set(
        SITES
    )

    while len(selected) < k:

        best = None
        best_gain = -np.inf

        for candidate in remaining:

            gain = (
                mean_IE[candidate]
            )

            for chosen in selected:

                gain += get_J(
                    candidate,
                    chosen
                )

            if gain > best_gain:

                best_gain = gain
                best = candidate

        selected.append(
            best
        )

        remaining.remove(
            best
        )

    return selected

In [ ]:
for k in [2, 3, 4, 6]:

    print(
        "\nk =", k
    )

    print(
        "AIE:",
        select_AIE(k)
    )

    print(
        "interaction:",
        select_interaction(k)
    )

In [ ]:
budgets = [
    2, 3, 4, 6
]

selected_AIE = {
    k: select_AIE(k)
    for k in budgets
}

selected_INT = {
    k: select_interaction(k)
    for k in budgets
}

In [ ]:
eval_rows = []

for c_idx, case in enumerate(
    test_cases
):

    (
        clean_states,
        corrupt_emb,
        D0
    ) = runtime_case(case)

    TE = case["TE"]

    for k in budgets:

        for method, selected in [
            (
                "AIE",
                selected_AIE[k]
            ),
            (
                "interaction",
                selected_INT[k]
            )
        ]:

            Dpatch = (
                patched_margin(
                    case,
                    clean_states,
                    corrupt_emb,
                    selected
                )
            )

            recovery = (
                Dpatch - D0
            ) / TE

            eval_rows.append({
                "case_id":
                    case["case_id"],

                "k": k,

                "method":
                    method,

                "recovery":
                    recovery,

                "restored_correct":
                    float(
                        Dpatch > 0
                    )
            })

    print(
        "holdout",
        c_idx + 1,
        "/",
        len(test_cases)
    )

In [ ]:
eval_df = pd.DataFrame(
    eval_rows
)

summary = (
    eval_df
    .groupby(
        ["k", "method"]
    )
    .agg(
        mean_recovery=(
            "recovery",
            "mean"
        ),

        median_recovery=(
            "recovery",
            "median"
        ),

        correct_rate=(
            "restored_correct",
            "mean"
        )
    )
)

print(summary)

In [ ]:
bootstrap_rows = []

rng = np.random.default_rng(
    999
)

for k in budgets:

    d = eval_df[
        eval_df["k"] == k
    ]

    pivot = d.pivot(
        index="case_id",
        columns="method",
        values="recovery"
    ).dropna()

    diff = (
        pivot["interaction"]
        -
        pivot["AIE"]
    ).to_numpy()

    boots = []

    for _ in range(5000):

        sample = rng.choice(
            diff,
            size=len(diff),
            replace=True
        )

        boots.append(
            sample.mean()
        )

    bootstrap_rows.append({
        "k": k,

        "delta_mean":
            diff.mean(),

        "CI_lo":
            np.percentile(
                boots,
                2.5
            ),

        "CI_hi":
            np.percentile(
                boots,
                97.5
            )
    })

benchmark_boot = pd.DataFrame(
    bootstrap_rows
)

print(
    benchmark_boot
)

In [ ]:
# Passiamo al terzo ordine.

In [ ]:
triple = [
    (10, "Q"),
    (8, "S"),
    (6, "Q")
]

third_rows = []

for case in train_cases:

    clean_states, corrupt_emb, D0 = (
        runtime_case(case)
    )

    TE = case["TE"]

    i, j, k = triple

    Di = patched_margin(
        case,
        clean_states,
        corrupt_emb,
        [i]
    )

    Dj = patched_margin(
        case,
        clean_states,
        corrupt_emb,
        [j]
    )

    Dk = patched_margin(
        case,
        clean_states,
        corrupt_emb,
        [k]
    )

    Dij = patched_margin(
        case,
        clean_states,
        corrupt_emb,
        [i, j]
    )

    Dik = patched_margin(
        case,
        clean_states,
        corrupt_emb,
        [i, k]
    )

    Djk = patched_margin(
        case,
        clean_states,
        corrupt_emb,
        [j, k]
    )

    Dijk = patched_margin(
        case,
        clean_states,
        corrupt_emb,
        [i, j, k]
    )

    J3 = (
        Dijk
        - Dij
        - Dik
        - Djk
        + Di
        + Dj
        + Dk
        - D0
    ) / TE

    # quanto il terzo sito aggiunge
    # realmente alla coppia?
    actual_increment = (
        Dijk - Dij
    ) / TE

    third_rows.append({
        "case_id":
            case["case_id"],

        "J3":
            J3,

        "actual_increment":
            actual_increment
    })

third_df = pd.DataFrame(
    third_rows
)

print(
    third_df.describe()
)

In [ ]:
print(
    "mean J3 =",
    third_df["J3"].mean()
)

print(
    "mean actual incremental effect =",
    third_df[
        "actual_increment"
    ].mean()
)

In [ ]:
CIRCUIT_SITES = [
    (10, "Q"),
    (8,  "S"),
    (6,  "Q"),
    (9,  "Q"),
    (7,  "S"),
    (2,  "Q"),
]

N_CIRCUIT = len(CIRCUIT_SITES)

for i, site in enumerate(CIRCUIT_SITES):
    print(i, site)

print("Subsets:", 2 ** N_CIRCUIT)

In [ ]:
def mask_to_sites(mask):

    return [
        CIRCUIT_SITES[i]
        for i in range(N_CIRCUIT)
        if mask & (1 << i)
    ]


def mask_order(mask):

    return int(mask).bit_count()


def mask_label(mask):

    if mask == 0:
        return "∅"

    return "+".join(
        f"L{l}{r}"
        for l, r in mask_to_sites(mask)
    )

In [ ]:
for mask in range(8):

    print(
        mask,
        format(mask, "06b"),
        mask_label(mask)
    )

In [ ]:
def evaluate_all_subsets(
    cases,
    save_path
):

    rows = []

    for c_idx, case in enumerate(cases):

        (
            clean_states,
            corrupt_emb,
            D0
        ) = runtime_case(case)

        TE = case["TE"]

        # insieme vuoto
        rows.append({
            "case_id": case["case_id"],
            "subset_mask": 0,
            "order": 0,
            "label": "∅",
            "recovery": 0.0
        })

        # tutti gli altri 63 subset
        for mask in range(
            1,
            2 ** N_CIRCUIT
        ):

            sites = mask_to_sites(
                mask
            )

            D = patched_margin(
                case,
                clean_states,
                corrupt_emb,
                sites
            )

            recovery = (
                D - D0
            ) / TE

            rows.append({
                "case_id":
                    case["case_id"],

                "subset_mask":
                    mask,

                "order":
                    mask_order(mask),

                "label":
                    mask_label(mask),

                "recovery":
                    recovery
            })

        pd.DataFrame(
            rows
        ).to_csv(
            save_path,
            index=False
        )

        print(
            f"{c_idx+1}/{len(cases)}"
            " complete"
        )

    return pd.DataFrame(rows)

In [ ]:
mobius_train_raw = evaluate_all_subsets(
    train_cases,
    "phase5_mobius_train_raw.csv"
)

In [ ]:
def mobius_transform(f):

    m = np.asarray(
        f,
        dtype=float
    ).copy()

    n = int(
        np.log2(len(m))
    )

    for bit in range(n):

        step = 1 << bit

        for mask in range(len(m)):

            if mask & step:

                m[mask] -= (
                    m[
                        mask ^ step
                    ]
                )

    return m

In [ ]:
def inverse_mobius(m):

    f = np.asarray(
        m,
        dtype=float
    ).copy()

    n = int(
        np.log2(len(f))
    )

    for bit in range(n):

        step = 1 << bit

        for mask in range(len(f)):

            if mask & step:

                f[mask] += (
                    f[
                        mask ^ step
                    ]
                )

    return f

In [ ]:
example_case = (
    mobius_train_raw[
        "case_id"
    ].iloc[0]
)

d = (
    mobius_train_raw[
        mobius_train_raw["case_id"]
        == example_case
    ]
    .sort_values(
        "subset_mask"
    )
)

f = d["recovery"].to_numpy()

m = mobius_transform(f)

f_reconstructed = (
    inverse_mobius(m)
)

print(
    "max reconstruction error:",
    np.max(
        np.abs(
            f - f_reconstructed
        )
    )
)

In [ ]:
def build_mobius_df(
    raw_df
):

    rows = []

    for case_id, d in (
        raw_df.groupby(
            "case_id"
        )
    ):

        d = d.sort_values(
            "subset_mask"
        )

        f = (
            d["recovery"]
            .to_numpy()
        )

        m = mobius_transform(
            f
        )

        for mask, coeff in enumerate(m):

            rows.append({
                "case_id":
                    case_id,

                "subset_mask":
                    mask,

                "order":
                    mask_order(mask),

                "label":
                    mask_label(mask),

                "mobius":
                    coeff
            })

    return pd.DataFrame(
        rows
    )

In [ ]:
mobius_train_raw = evaluate_all_subsets(
    train_cases,
    "phase5_mobius_train_raw.csv"
)

In [ ]:
mobius_test_raw = evaluate_all_subsets(
    test_cases,
    "phase5_mobius_test_raw.csv"
)

In [ ]:
mobius_train = build_mobius_df(
    mobius_train_raw
)

mobius_test = build_mobius_df(
    mobius_test_raw
)

In [ ]:
def order_profile(
    mobius_df
):

    per_case = (
        mobius_df
        .groupby(
            [
                "case_id",
                "order"
            ]
        )
        .agg(
            signed_sum=(
                "mobius",
                "sum"
            ),

            abs_sum=(
                "mobius",
                lambda x:
                    np.abs(x).sum()
            )
        )
        .reset_index()
    )

    summary = (
        per_case
        .groupby("order")
        .agg(
            mean_signed=(
                "signed_sum",
                "mean"
            ),

            std_signed=(
                "signed_sum",
                "std"
            ),

            mean_abs=(
                "abs_sum",
                "mean"
            ),

            std_abs=(
                "abs_sum",
                "std"
            )
        )
    )

    return (
        per_case,
        summary
    )

In [ ]:
train_order_cases, train_order = (
    order_profile(
        mobius_train
    )
)

test_order_cases, test_order = (
    order_profile(
        mobius_test
    )
)

print("TRAIN")
print(train_order)

print("\nTEST")
print(test_order)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    train_order.index,
    train_order[
        "mean_abs"
    ],
    marker="o",
    label="train"
)

plt.plot(
    test_order.index,
    test_order[
        "mean_abs"
    ],
    marker="o",
    label="holdout"
)

plt.xlabel(
    "Interaction order"
)

plt.ylabel(
    "Mean absolute Möbius mass"
)

plt.title(
    "Causal interaction order spectrum"
)

plt.legend()

plt.show()

In [ ]:
def truncated_reconstruction(
    m,
    max_order
):

    n = int(
        np.log2(len(m))
    )

    f_hat = np.zeros_like(
        m,
        dtype=float
    )

    for mask in range(
        1 << n
    ):

        sub = mask

        while True:

            if (
                mask_order(sub)
                <= max_order
            ):

                f_hat[mask] += (
                    m[sub]
                )

            if sub == 0:
                break

            sub = (
                sub - 1
            ) & mask

    return f_hat

In [ ]:
def truncation_errors(
    raw_df
):

    rows = []

    for case_id, d in (
        raw_df.groupby(
            "case_id"
        )
    ):

        d = d.sort_values(
            "subset_mask"
        )

        f = (
            d["recovery"]
            .to_numpy()
        )

        m = mobius_transform(f)

        for k in range(
            1,
            N_CIRCUIT + 1
        ):

            f_hat = (
                truncated_reconstruction(
                    m,
                    k
                )
            )

            mae = np.mean(
                np.abs(
                    f - f_hat
                )
            )

            max_error = np.max(
                np.abs(
                    f - f_hat
                )
            )

            rows.append({
                "case_id":
                    case_id,

                "max_order":
                    k,

                "MAE":
                    mae,

                "max_error":
                    max_error
            })

    return pd.DataFrame(
        rows
    )

In [ ]:
trunc_train = truncation_errors(
    mobius_train_raw
)

trunc_test = truncation_errors(
    mobius_test_raw
)

In [ ]:
print(
    trunc_train
    .groupby(
        "max_order"
    )
    ["MAE"]
    .mean()
)

print()

print(
    trunc_test
    .groupby(
        "max_order"
    )
    ["MAE"]
    .mean()
)

In [ ]:
train_mae = (
    trunc_train
    .groupby(
        "max_order"
    )
    ["MAE"]
    .mean()
)

test_mae = (
    trunc_test
    .groupby(
        "max_order"
    )
    ["MAE"]
    .mean()
)

plt.figure(
    figsize=(8,5)
)

plt.plot(
    train_mae.index,
    train_mae.values,
    marker="o",
    label="train"
)

plt.plot(
    test_mae.index,
    test_mae.values,
    marker="o",
    label="holdout"
)

plt.xlabel(
    "Maximum Möbius order retained"
)

plt.ylabel(
    "Mean reconstruction error"
)

plt.title(
    "How many causal orders are necessary?"
)

plt.legend()

plt.show()

In [ ]:
train_subset_mean = (
    mobius_train_raw
    .groupby(
        "subset_mask"
    )
    ["recovery"]
    .mean()
)

In [ ]:
exact_best = {}

for k in range(
    1,
    N_CIRCUIT + 1
):

    candidate_masks = [
        mask
        for mask in range(
            1 << N_CIRCUIT
        )
        if mask_order(mask) == k
    ]

    best_mask = max(
        candidate_masks,
        key=lambda m:
            train_subset_mean[m]
    )

    exact_best[k] = best_mask

    print(
        f"k={k}",
        "|",
        mask_label(best_mask),
        "| train recovery =",
        train_subset_mean[
            best_mask
        ]
    )

In [ ]:
exact_test_rows = []

for k, mask in exact_best.items():

    vals = (
        mobius_test_raw[
            mobius_test_raw[
                "subset_mask"
            ] == mask
        ]["recovery"]
    )

    exact_test_rows.append({
        "k": k,

        "circuit":
            mask_label(mask),

        "test_mean":
            vals.mean(),

        "test_median":
            vals.median(),

        "test_std":
            vals.std()
    })

exact_test = pd.DataFrame(
    exact_test_rows
)

print(exact_test)